In [6]:
import pandas as pd

# Đọc dữ liệu từ tệp CSV

def clean_data(file_path, output_path):
    df = pd.read_csv(file_path)
    
    # Kiểm tra số lượng giá trị bị thiếu ở mỗi cột
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percentage})
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    print("Số lượng giá trị bị thiếu ở mỗi cột:")
    print(missing_data.sort_values(by='Missing Values', ascending=False))
    
    # Xóa các cột không cần thiết trước
    columns_to_drop = ['Restaurant penalty (Rejection)', 'Review', 'Instructions']
    df = df.drop(columns=columns_to_drop, errors='ignore')
    print("Các cột đã bị xóa:", columns_to_drop)
    
    # Loại bỏ các dòng trùng lặp
    df = df.drop_duplicates()
    
    # Chuyển đổi cột 'Order Placed At' thành kiểu datetime
    df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y', errors='coerce')
    
    # Chuyển đổi 'Distance' thành số (loại bỏ km, xử lý "<1km" thành 0.5)
    df['Distance'] = df['Distance'].str.replace('km', '', regex=True)
    df['Distance'] = df['Distance'].replace('<1', 0.5).astype(float)
    
    # Điền giá trị "None" cho các ô trống trong các cột yêu cầu
    df['Cancellation / Rejection reason'] = df['Cancellation / Rejection reason'].fillna("None")
    df['Customer complaint tag'] = df['Customer complaint tag'].fillna("None")
    
    # Xác định các hàng bị loại bỏ
    threshold = 0.2 * df.shape[1]  # Hơn 20% số cột bị thiếu dữ liệu
    dropped_rows = df[df.isnull().sum(axis=1) > threshold].index.tolist()
    
    # Loại bỏ các hàng có hơn 20% giá trị bị thiếu
    df = df.dropna(thresh=threshold, axis=0)
    
    print("Các hàng bị loại bỏ:", len(dropped_rows))
    
    # Lưu kết quả vào một tệp CSV mới
    df.to_csv(output_path, index=False)
    print(f"Dữ liệu đã được làm sạch và lưu vào: {output_path}")
    
    return df

# Đường dẫn đến tệp dữ liệu đầu vào và đầu ra
file_path = "order_history_kaggle_data.csv"
output_path = "cleaned_order_history.csv"
cleaned_df = clean_data(file_path, output_path)


Số lượng giá trị bị thiếu ở mỗi cột:
                                        Missing Values  Percentage
Restaurant penalty (Rejection)                   21318   99.985929
Restaurant compensation (Cancellation)           21188   99.376202
Cancellation / Rejection reason                  21135   99.127621
Review                                           21025   98.611697
Customer complaint tag                           20852   97.800291
Instructions                                     20601   96.623048
Rating                                           18830   88.316683
Discount construct                                5498   25.786783
KPT duration (minutes)                             295    1.383612
Rider wait time (minutes)                          168    0.787956
Các cột đã bị xóa: ['Restaurant penalty (Rejection)', 'Review', 'Instructions']
Các hàng bị loại bỏ: 0
Dữ liệu đã được làm sạch và lưu vào: cleaned_order_history.csv
